# Customer Support AI using an SLM

Fine-tuning **Qwen2.5-0.5B-Instruct** with LoRA on the
[`talkmap/banking-conversation-corpus`](https://huggingface.co/datasets/talkmap/banking-conversation-corpus)
dataset, to build a customer-support response generator.

Run this notebook top-to-bottom (Run All). Expect the training cell to take
roughly 1.5-3 hours on a Kaggle T4 GPU.

## 1. Install dependencies

`torchao` must be upgraded *before* `peft` is used, or LoRA adapter injection fails with an `ImportError`.

In [ ]:
!pip install -q -U torchao
!pip install -q datasets transformers accelerate peft bitsandbytes trl
!pip install -q gradio rouge-score sacrebleu bert-score

## 2. Load the dataset

The corpus has 5.5M+ rows (one row per message). We stream it and take a 200K-row slice — loading the full split directly caused Kaggle's kernel to OOM.

In [ ]:
from datasets import load_dataset, Dataset

ds_stream = load_dataset("talkmap/banking-conversation-corpus", split="train", streaming=True)
subset = list(ds_stream.take(200000))
ds = Dataset.from_list(subset)
print(ds)

## 3. Reconstruct conversations into training pairs

Each row is a single message (`conversation_id`, `speaker`, `date_time`, `text`).
We group by conversation, sort by time, and for every **agent** turn that follows
at least one **client** message, create one training example:

- `context` — the full conversation so far (all prior turns)
- `query` — the most recent client message
- `response` — the agent's reply (training target)

Rows with a missing/`None` `text` field (present in the raw data) are dropped.

In [ ]:
import pandas as pd

df = ds.to_pandas()
df["date_time"] = pd.to_datetime(df["date_time"], format="ISO8601")
df = df.sort_values(["conversation_id", "date_time"])

examples = []
for conv_id, group in df.groupby("conversation_id", sort=False):
    turns = group[["speaker", "text"]].to_dict("records")
    history = []
    for turn in turns:
        if turn["speaker"] == "agent" and history:
            if any(h["speaker"] == "client" for h in history):
                context_str = "\n".join(f"{h['speaker'].capitalize()}: {h['text']}" for h in history)
                last_client_msg = next(h["text"] for h in reversed(history) if h["speaker"] == "client")
                examples.append({
                    "conversation_id": conv_id,
                    "context": context_str,
                    "query": last_client_msg,
                    "response": turn["text"],
                })
        history.append(turn)

# Drop rows with missing/null text fields (a handful exist in the raw corpus)
examples = [ex for ex in examples if ex.get("context") and ex.get("query") and ex.get("response")]
print(f"Reconstructed examples: {len(examples)}")

## 4. Tokenizer, system prompt, and chat formatting

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

SYSTEM_PROMPT = (
    "You are a professional, empathetic customer support agent for a bank. "
    "Respond helpfully and concisely to the customer's message, using the "
    "conversation history for context."
)

def format_example(ex):
    user_content = f"{ex['context']}\n\nRespond as the Agent to the most recent Client message."
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": ex["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

## 5. Filter by length, split, and save to disk

We measured the token-length distribution of this dataset ahead of time
(median ~423, p90 ~744, p99 ~1674). We cap at **1024 tokens** and drop the
long tail rather than truncating (truncating risks cutting off the agent's
response mid-sentence, which is a bad training signal).

We also save the resulting train/eval splits to `/kaggle/working/` immediately —
this reconstruction step is the most expensive to redo if a session drops.

In [ ]:
import random
import json

MAX_SEQ_LEN = 1024

def get_length(ex):
    return len(tokenizer(format_example(ex))["input_ids"])

filtered = [ex for ex in examples if get_length(ex) <= MAX_SEQ_LEN]
print(f"Kept {len(filtered)} / {len(examples)} examples after filtering")

random.seed(42)
random.shuffle(filtered)

TRAIN_SIZE = 4000
EVAL_SIZE = 500
train_examples = filtered[:TRAIN_SIZE]
eval_examples = filtered[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE]
print(f"Train: {len(train_examples)}, Eval: {len(eval_examples)}")

with open("/kaggle/working/train_examples.json", "w") as f:
    json.dump(train_examples, f)
with open("/kaggle/working/eval_examples.json", "w") as f:
    json.dump(eval_examples, f)
print("Saved train/eval splits to /kaggle/working/")

## 6. Load the base model and attach a LoRA adapter

LoRA freezes all original weights and injects small trainable adapter
matrices into the attention projections (`q_proj`, `k_proj`, `v_proj`,
`o_proj`). Only the adapters are trained — typically under 1% of total
parameters — which is what makes LoRA far cheaper than full fine-tuning.

We use plain LoRA (bf16), not QLoRA: QLoRA's 4-bit quantization exists to
fit *large* models (7B+) into limited memory. At 0.5B, plain bf16 LoRA
already fits comfortably in a T4's 16GB, so quantization would only add
complexity and a small quality cost for no real benefit here.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,               # rank of the adapter bottleneck
    lora_alpha=32,       # scaling factor, conventionally 2x r
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Tokenize with loss masking

For each example we tokenize the prompt (system+user) and the full sequence
(system+user+assistant) separately, then set every prompt-token's label to
`-100` (PyTorch's cross-entropy "ignore" value). This means the model is only
ever scored on generating the agent's response — not on "predicting" the
context/query it's given as input.

Note: `tokenizer.apply_chat_template(..., tokenize=True)` returns a
`BatchEncoding`, not a plain list — we pull out `["input_ids"]` explicitly.

In [ ]:
def tokenize_example(ex):
    system_msg = {"role": "system", "content": SYSTEM_PROMPT}
    user_msg = {"role": "user", "content": f"{ex['context']}\n\nRespond as the Agent to the most recent Client message."}
    assistant_msg = {"role": "assistant", "content": ex["response"]}

    prompt_ids = tokenizer.apply_chat_template(
        [system_msg, user_msg], tokenize=True, add_generation_prompt=True, return_tensors=None
    )["input_ids"]
    full_ids = tokenizer.apply_chat_template(
        [system_msg, user_msg, assistant_msg], tokenize=True, add_generation_prompt=False, return_tensors=None
    )["input_ids"]

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

train_ds = Dataset.from_list(train_examples).map(
    tokenize_example, remove_columns=["conversation_id", "context", "query", "response"]
)
eval_ds = Dataset.from_list(eval_examples).map(
    tokenize_example, remove_columns=["conversation_id", "context", "query", "response"]
)
print("Tokenized train/eval ready:", len(train_ds), len(eval_ds))

# Sanity check: decoded unmasked labels should exactly match the ground-truth response
sample = train_ds[0]
non_masked = [l for l in sample["labels"] if l != -100]
print("Decoded response portion:", tokenizer.decode(non_masked))

## 8. Train

- **Batch size 4 + gradient accumulation 4** → effective batch size 16 (fits a T4's 16GB at 1024-token sequences).
- **Learning rate 2e-4** — a standard starting point for LoRA (tolerates a higher LR than full fine-tuning since only a small adapter is updated).
- **3 epochs over 4000 examples** — enough to converge without overfitting a small dataset.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import time

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen-support-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

start = time.time()
trainer.train()
train_time = time.time() - start
print(f"Training time: {train_time/60:.1f} minutes")

## 9. Save the fine-tuned adapter

This is the `model/` folder required in the submission zip.

In [ ]:
FINAL_MODEL_DIR = "/kaggle/working/model"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved final adapter to", FINAL_MODEL_DIR)

## 10. Task 2 — Evaluate: Base SLM vs Fine-tuned SLM

Load a fresh, untouched base model alongside the fine-tuned model (base + LoRA
adapter loaded from disk) for a fair comparison.

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto"),
    FINAL_MODEL_DIR
)
print("Base and fine-tuned models loaded.")

In [ ]:
def generate_response(model, ex, max_new_tokens=150):
    system_msg = {"role": "system", "content": SYSTEM_PROMPT}
    user_msg = {"role": "user", "content": f"{ex['context']}\n\nRespond as the Agent to the most recent Client message."}

    inputs = tokenizer.apply_chat_template(
        [system_msg, user_msg], tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy decoding for a fair, reproducible comparison
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
N_EVAL = 50  # subset of the 500 held-out eval examples

results = []
start = time.time()
for i, ex in enumerate(eval_examples[:N_EVAL]):
    base_resp = generate_response(base_model, ex)
    ft_resp = generate_response(finetuned_model, ex)
    results.append({
        "query": ex["query"],
        "ground_truth": ex["response"],
        "base_response": base_resp,
        "finetuned_response": ft_resp,
    })
    if (i + 1) % 10 == 0:
        print(f"{i+1}/{N_EVAL} done...")

gen_time = time.time() - start
print(f"\nGeneration time for {N_EVAL} examples (both models): {gen_time/60:.1f} minutes")

with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved eval_results.json")

### Automated metrics: ROUGE, BLEU, BERTScore

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bert_score

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge_bleu(responses, references):
    rouge1, rouge2, rougeL = [], [], []
    for resp, ref in zip(responses, references):
        scores = rouge.score(ref, resp)
        rouge1.append(scores["rouge1"].fmeasure)
        rouge2.append(scores["rouge2"].fmeasure)
        rougeL.append(scores["rougeL"].fmeasure)
    bleu = sacrebleu.corpus_bleu(responses, [references]).score
    return {
        "rouge1": sum(rouge1) / len(rouge1),
        "rouge2": sum(rouge2) / len(rouge2),
        "rougeL": sum(rougeL) / len(rougeL),
        "bleu": bleu,
    }

references = [r["ground_truth"] for r in results]
base_responses = [r["base_response"] for r in results]
ft_responses = [r["finetuned_response"] for r in results]

print("=== BASE MODEL ===")
base_metrics = compute_rouge_bleu(base_responses, references)
print(base_metrics)

print("\n=== FINE-TUNED MODEL ===")
ft_metrics = compute_rouge_bleu(ft_responses, references)
print(ft_metrics)

print("\nComputing BERTScore...")
P_base, R_base, F1_base = bert_score(base_responses, references, lang="en", verbose=False)
P_ft, R_ft, F1_ft = bert_score(ft_responses, references, lang="en", verbose=False)
print(f"Base BERTScore F1: {F1_base.mean().item():.4f}")
print(f"Fine-tuned BERTScore F1: {F1_ft.mean().item():.4f}")

metrics_summary = {
    "base": {**base_metrics, "bertscore_f1": F1_base.mean().item()},
    "finetuned": {**ft_metrics, "bertscore_f1": F1_ft.mean().item()},
}
with open("/kaggle/working/metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("\nSaved metrics_summary.json")
print(metrics_summary)

### Manual inspection

Print all generated pairs to select 5-10 representative examples (a mix of wins and honest failure cases) for the README.

In [ ]:
for i, r in enumerate(results):
    print(f"--- Example {i} ---")
    print("QUERY:", r["query"][:150])
    print("GROUND TRUTH:", r["ground_truth"][:200])
    print("BASE:", r["base_response"][:200])
    print("FINE-TUNED:", r["finetuned_response"][:200])
    print()

## 11. Task 3 — Write the Gradio UI (`app.py`)

Written as a standalone, portable script: it loads the model from a
**relative** `./model` path, so it works correctly wherever the submission
zip is unzipped and run — not just inside this Kaggle session.

In [ ]:
%%writefile /kaggle/working/app.py
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_DIR = "./model"  # relative path — model/ folder must sit next to this script

SYSTEM_PROMPT = (
    "You are a professional, empathetic customer support agent for a bank. "
    "Respond helpfully and concisely to the customer\'s message, using the "
    "conversation history for context."
)

print("Loading model, please wait...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.eval()
print("Model loaded.")


def generate_response(customer_message, context):
    context = (context or "").strip()
    if context:
        user_content = f"{context}\nClient: {customer_message}\n\nRespond as the Agent to the most recent Client message."
    else:
        user_content = f"Client: {customer_message}\n\nRespond as the Agent to the most recent Client message."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


with gr.Blocks(title="Customer Support AI") as demo:
    gr.Markdown("## \U0001F4AC Customer Support AI")
    customer_message = gr.Textbox(label="Customer Message", placeholder="My card hasn\'t arrived yet...")
    context = gr.Textbox(
        label="Previous Conversation / Context (optional)",
        placeholder="Agent: Hi, how can I help?\nClient: ...",
        lines=4,
    )
    generate_btn = gr.Button("Generate Response")
    response = gr.Textbox(label="AI Agent Response", lines=4, interactive=False)

    generate_btn.click(fn=generate_response, inputs=[customer_message, context], outputs=response)

if __name__ == "__main__":
    demo.launch()

## 12. Done

At this point `/kaggle/working/` contains everything needed for submission:
`model/` (the LoRA adapter), `app.py`, and the JSON files with your saved
results/metrics for writing up the README. Download these before your Kaggle
session ends — Draft Sessions are not persisted across restarts unless you
Quick Save or Commit.